In [ ]:
import torch
from torchvision.transforms import v2
from medmnist import PathMNIST

tf = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True)
])

val_dataset = PathMNIST(root="./data/",split="val",transform=tf,download=True,size=64)

n_labels = len(val_dataset.info["label"].items())

In [ ]:
import os
from models.mae import MaskedAutoEncoder,AutoEncoder
from models.cross_predictor_hybrid import Predictor


model_type = "cross_predictor_hybrid"
model_id = "2.2"

notes = "better augmentation that fixes rotation black corners"

model_save_folder = f"model_weights/checkpoints/{model_type}/{model_id}"

os.makedirs(model_save_folder,exist_ok=True)

freeze_encoder = True
is_linear_probe = False

mae = MaskedAutoEncoder()
mae.load_state_dict(torch.load("model_weights/pretrain_checkpoints/model_13_epoch_800.pt"))
ae = AutoEncoder(patcher=mae.patcher,encoder=mae.encoder)
model = Predictor(autoencoder=ae,n_labels=n_labels)

if freeze_encoder == True:
    for name, param in model.autoencoder.named_parameters():
        param.requires_grad = False

print(model)


In [ ]:
model_no_parameters = sum(param.numel() for param in model.parameters())
print(f"Total Parameters in {model_type} : {model_no_parameters}")

In [ ]:
if is_linear_probe:
    tf_data_aug = v2.Compose([
        v2.ToImage(),
        v2.RandomHorizontalFlip(0.5),
        v2.RandomVerticalFlip(0.5),
        v2.Pad(padding=16,padding_mode="reflect"),
        v2.RandomRotation(degrees=(0,360),interpolation=v2.InterpolationMode.BILINEAR),
        v2.CenterCrop(size=(64,64)),
        v2.ToDtype(torch.float32, scale=True),
    ])
else:
    tf_data_aug = v2.Compose([
        v2.ToImage(),
        v2.RandomHorizontalFlip(0.5),
        v2.RandomVerticalFlip(0.5),
        v2.Pad(padding=16,padding_mode="reflect"),
        v2.RandomRotation(degrees=(0,360),interpolation=v2.InterpolationMode.BILINEAR),
        v2.CenterCrop(size=(64,64)),
        v2.ToDtype(torch.float32, scale=True),
        v2.ColorJitter(
            brightness=0.3,
            contrast=0.3,
            saturation=0.3,
            hue=0.01
        )
    ])


train_dataset = PathMNIST(root="./data/",split="train",transform=tf_data_aug,download=True,size=64)

In [ ]:
from torch.utils.data import DataLoader

num_workers = 4

train_batch_size = 256
val_batch_size = 256
effective_batch = 256
if 256 % train_batch_size != 0:
    raise ValueError()

train_dl = DataLoader(
    train_dataset,
    batch_size= train_batch_size,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=True,
    persistent_workers=True
)

val_dl = DataLoader(
    val_dataset,
    batch_size = val_batch_size,
    num_workers=num_workers,
    pin_memory=True,
    persistent_workers=True
)


In [ ]:
import math
from torch import nn
from torch import optim

grad_acc = max(1,(256//train_batch_size))
steps_per_epoch = len(train_dl)

epochs = 100
warm_up_epochs = math.ceil(epochs * 0.025)
checkpointing_rate = 10

warm_up_steps = warm_up_epochs*steps_per_epoch
cosine_steps = (epochs - warm_up_epochs)*steps_per_epoch

base_lr = 1e-4
encoder_base_lr = 1e-5
max_lr = base_lr * (effective_batch/256)
encoder_max_lr = encoder_base_lr * (effective_batch/256)
min_lr = 1e-6

betas = (0.9, 0.999)
weight_decay = 0.01

if freeze_encoder:
    optimiser = optim.AdamW([
                {"params":model.predictor.parameters(),"lr":max_lr,"betas":betas,"weight_decay":weight_decay},
            ])
else:
    optimiser = optim.AdamW([
            {"params":model.autoencoder.parameters(),"lr":encoder_max_lr,"betas":betas,"weight_decay":weight_decay},
            {"params":model.predictor.parameters(),"lr":max_lr,"betas":betas,"weight_decay":weight_decay},
        ])
loss = nn.CrossEntropyLoss(label_smoothing=0.1)

# start at minimum and go up
warmup_scheduler = optim.lr_scheduler.LinearLR(
    optimiser,
    start_factor=0.1,
    end_factor=1.0,
    total_iters=warm_up_steps
)
cosine_scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimiser,
    T_max=cosine_steps,
    eta_min=min_lr
)
scheduler = optim.lr_scheduler.SequentialLR(
    optimiser,
    schedulers=[warmup_scheduler, cosine_scheduler],
    milestones=[warm_up_steps]
)


In [ ]:
import os
from training_functions import train,test
from tqdm.notebook import tqdm
import mlflow
import time
import csv

log_folder_path = f"logs/{model_type}/{model_id}"

os.makedirs(log_folder_path,exist_ok=True)

log_file_path = f"{log_folder_path}/training_log_{int(time.time())}.csv"

device = "cuda" if torch.cuda.is_available() else "cpu"

mlflow.set_tracking_uri("http://192.168.1.99:5000")
mlflow.set_experiment(model_type)
mlflow.set_experiment_tags({
    "stage":"probe_training"
})
with mlflow.start_run(run_name=model_id):

    mlflow_dataset = mlflow.data.numpy_dataset.from_numpy(
        features=train_dataset.imgs,
        targets=train_dataset.labels
    )
    mlflow.log_input(mlflow_dataset, context="training")

    mlflow.log_params({
        "training_batch_size":train_batch_size,
        "effective_batch":effective_batch,
        "base_lr":base_lr,
        "max_lr":max_lr,
        "min_lr":min_lr,
        "epochs":epochs,
        "warmup_epochs":warm_up_epochs,
        "steps_per_epoch":steps_per_epoch,
        "gradient_accumulation":grad_acc,
        "frozen_encoder":freeze_encoder,
        "parameter_count":model_no_parameters,
    })

    mlflow.set_tags({
        "model_id": model_id,
        "notes":notes,
    })

    mlflow.log_text(str(tf_data_aug),"training_augmentations.txt")
    mlflow.log_text(str(optimiser),"optimiser_config.txt")
    mlflow.log_text(str(scheduler.__dict__),"scheduler_config.txt")
    loss_config = {
        "loss_type":type(loss).__name__,
        "label_smoothing":getattr(loss,"label_smoothing",0.0),
        "ignore_index": getattr(loss, "ignore_index", -100),
        "reduction": getattr(loss, "reduction", "mean"),
    }
    mlflow.log_text(str(loss_config),"loss_config.txt")
    

    with open(log_file_path, mode="w", newline="") as f:
        prev_train_loss = None
        prev_val_loss = None

        writer = csv.writer(f)
        csv_header = ["epoch", "train_loss", "val_loss","train_delta","val_delta","train_acc","val_acc","train_auc","val_auc","predictor_lr"]
        if freeze_encoder:
            writer.writerow(csv_header)
        else:
            csv_header.append("encoder_lr")
            writer.writerow(csv_header)

        model = model.to(device)
        torch.set_float32_matmul_precision('high')
        model = torch.compile(model)

        with tqdm(range(epochs),desc="Epochs") as bar:
            for epoch in bar:
                
                train_delta = val_delta = 0.0

                train_loss,train_acc,train_auc = train(model, device, train_dl, loss, optimiser, epoch, scheduler, grad_acc)

                if freeze_encoder:
                    current_predictor_lr = optimiser.param_groups[0]['lr']
                else:
                    current_encoder_lr = optimiser.param_groups[0]['lr']
                    current_predictor_lr = optimiser.param_groups[1]['lr']

                val_loss,val_acc,val_auc = test(model, device, val_dl, loss,epoch)

                train_delta = train_loss - prev_train_loss  if prev_train_loss is not None else 0.0
                val_delta = val_loss - prev_val_loss if prev_val_loss is not None else 0.0

                prev_train_loss = train_loss
                prev_val_loss = val_loss

                mlflow_metrics = {
                    "train_loss":train_loss,
                    "val_loss":val_loss,
                    "train_delta":train_delta,
                    "val_delta":val_delta,
                    "train_acc":train_acc,
                    "val_acc":val_acc,
                    "train_auc":train_auc,
                    "val_auc":val_auc,
                    "predictor_lr":current_predictor_lr,
                }
                if not freeze_encoder:
                    mlflow_metrics.update({"encoder_lr": current_encoder_lr})

                postfix_metrics = { k:f"{v:.3g}" for k,v in mlflow_metrics.items()}
                
                bar.set_postfix(postfix_metrics)
                mlflow.log_metrics(mlflow_metrics,step=(epoch+1))

                csv_arr = [epoch+1,train_loss,val_loss,train_delta,val_delta,train_acc,val_acc,train_auc,val_auc,current_predictor_lr]
                if freeze_encoder:
                    writer.writerow(csv_arr)
                else:
                    csv_arr.append(current_predictor_lr)
                    writer.writerow(csv_arr)
                f.flush()

                if (epoch+1) % checkpointing_rate == 0:
                    torch.save(model._orig_mod.state_dict() if hasattr(model, '_orig_mod') else model.state_dict(),f"{model_save_folder}/model_epoch_{epoch+1}.pt")

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import time

graph_save_folder = f"graphs/training/{model_type}/{model_id}"

os.makedirs(graph_save_folder,exist_ok=True)

dataframe = pd.read_csv(log_file_path)

plt.plot(dataframe["epoch"],dataframe["train_loss"])
plt.plot(dataframe["epoch"],dataframe["val_loss"])
plt.title
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.savefig(f"{graph_save_folder}/train_loss_{int(time.time())}.png")
plt.show()
plt.close()